# Setup

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

X = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/train.csv', index_col='id') 
X_test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/test.csv', index_col='id')

X.head()

# Remove rows with missing target, separate target from predictors
X.dropna(axis=0, subset=['satisfaction'], inplace=True)
y = X.satisfaction.astype(int)
X.drop(['satisfaction'], axis=1, inplace=True)

# Seperate validation & training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=0.8, test_size=0.2,
                                                      random_state=0)

X_train.head()


,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,...,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class
id,,,,,,,,,,,,,,,,,,,,,
661386,41,2342,2,2,2,2,5,4,4,5,...,5,5,3,5,0,0.0,Male,Loyal Customer,Business travel,Business
539051,24,1670,2,3,3,3,2,2,2,2,...,1,2,4,2,0,0.0,Male,Loyal Customer,Business travel,Business
94434,41,2235,1,1,1,1,4,4,4,3,...,3,3,3,5,0,0.0,Female,Loyal Customer,Business travel,Business
137187,37,634,5,5,5,5,4,4,4,4,...,4,4,3,5,0,0.0,Male,Loyal Customer,Business travel,Business
580346,57,2139,4,5,4,4,3,5,5,5,...,5,5,4,4,0,0.0,Male,Loyal Customer,Business travel,Business


In [ ]:
# Find missing-values columns
X.isnull().sum()

# Cross-validation: Pipeline including One-Hot Encoding & Imputation 

In [24]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Identify categorical and numerical columns directly from X
categorical_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
numerical_cols = X.select_dtypes(exclude=['object', 'bool']).columns.tolist()

,Age,Flight_Distance,Inflight_wifi_service,Departure/Arrival_time_convenient,Ease_of_Online_booking,Gate_location,Food_and_drink,Online_boarding,Seat_comfort,Inflight_entertainment,...,Arrival_Delay_in_Minutes,Gender_Female,Gender_Male,Customer_Type_Loyal_Customer,Customer_Type_disloyal_Customer,Type_of_Travel_Business_travel,Type_of_Travel_Personal_Travel,Class_Business,Class_Eco,Class_Eco_Plus
id,,,,,,,,,,,,,,,,,,,,,
661386,41.0,2342.0,2.0,2.0,2.0,2.0,5.0,4.0,4.0,5.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
539051,24.0,1670.0,2.0,3.0,3.0,3.0,2.0,2.0,2.0,2.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
94434,41.0,2235.0,1.0,1.0,1.0,1.0,4.0,4.0,4.0,3.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
137187,37.0,634.0,5.0,5.0,5.0,5.0,4.0,4.0,4.0,4.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
580346,57.0,2139.0,4.0,5.0,4.0,4.0,3.0,5.0,5.0,5.0,...,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0


In [9]:
# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

def create_model(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1
):
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=0,
        n_jobs=-1
    )

    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

Accuracy: 0.9249751656220744


In [ ]:
# First accuracy scoring
from sklearn.metrics import accuracy_score

# Create model
clf = create_model()

# Train
clf.fit(X_train, y_train)

# Predict
preds = clf.predict(X_valid)

# Evaluate
print('Accuracy:', accuracy_score(y_valid, preds))

To test optimization strategies, we advanced to utilizing a joint function for all optimization strategies:

In [ ]:
from sklearn.model_selection import cross_val_score

def get_score(**params):

    clf = create_model(**params)

    scores = cross_val_score(
        clf,
        X,
        y,
        cv=3,
        scoring='accuracy'
    )

    return scores.mean()

# Optimizing the model

## Number of estimators / decision trees

Our test shows extremely diminishing returns; 50 trees are basically enough.

Results:
```
50 trees: 0.9242
100 trees: 0.9247
200 trees: 0.9249
300 trees: 0.9250
```

`n_estimators` should be **100** for now, but choosing any amount higher than 50 shows only minor improvements.

We will now try other methods of optimizing our model.

## Maximum depth

Results show diminishing returns:
```
max_depth=10: 0.9178
max_depth=15: 0.9241
max_depth=20: 0.9254
max_depth=25: 0.9254
max_depth=30: 0.9250
max_depth=None: 0.9247
```
We will set `max_depth` to **20**.

## Minimum samples split

Results show diminishing returns:
```
min_samples_split=2: 0.9247
min_samples_split=5: 0.9254
min_samples_split=10: 0.9257
min_samples_split=20: 0.9257
```
We will set `min_samples_split` to **10**.

## Minimum samples leaf

Results show decreasing returns after an amount of 2:
```
min_samples_leaf=1: 0.9247
min_samples_leaf=2: 0.9255
min_samples_leaf=4: 0.9254
min_samples_leaf=8: 0.9251
min_samples_leaf=16: 0.9246
```
We will set `min_samples_leaf` to 2.

# Defining our definitive model
Parameters:
- `n_estimators`: **100**
- `max_depth`: **20**
- `min_samples_split`: **10**
- `min_samples_leaf`: 2

In [ ]:
final_model = create_model(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=2
)

final_model.fit(X, y)

test_predictions = final_model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Evaluate inal model
final_cv_scores = cross_val_score(final_model, X, y, cv=3, scoring='accuracy')

print("Final Model Statistics:")
print(f"Cross-Validation Accuracy Scores: {final_cv_scores}")
print(f"Mean CV Accuracy: {final_cv_scores.mean():.4f}")
print(f"Standard Deviation: {final_cv_scores.std():.4f}")

# Generate the Kaggle submission file
output = pd.DataFrame({
    'id': X_test.index,
    'satisfaction': test_predictions
})

output.to_csv('submission.csv', index=False)
print("\nSubmission file 'submission.csv' successfully created and saved!")

## Final Results
Final Model Statistics:
- Cross-Validation Accuracy Scores: `[0.92640602 0.92500386 0.92504213]`
- Mean CV Accuracy: `0.9255`
- Standard Deviation: `0.0007`
